# Retirement figures

Figures for `Text/RetirementModel.tex`, written to `Figures/Figures_v4`.
The solver is `Endogenous-Retirement-Search-v4.py`. Every consumption
rule is the worker's $C(m_t)$ at $t = 15$ of a $T = 20$ problem.

Three blocks, in run order:

1. **Paper figures.** The four consumption figures in the retirement section.
2. **Direct comparison.** Choose the search cost and the account dispersion to match the Extreme Value consumption function at $\sigma_\zeta = 0.10$.
3. **Additional figures.** Not included in the text.

## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import importlib.util
from pathlib import Path

module_path = Path("Endogenous-Retirement-Search-v4.py")
spec = importlib.util.spec_from_file_location("retirement_search", module_path)
retirement_search = importlib.util.module_from_spec(spec)
spec.loader.exec_module(retirement_search)

RetiringDeaton = retirement_search.RetiringDeaton

fig_dir = Path("../Figures/Figures_v4")
fig_dir.mkdir(parents=True, exist_ok=True)


def _series(models, colors, linestyles, t):
    series = []
    for model, color, ls in zip(models, colors, linestyles):
        m = model.mGrid
        c = model.solution[t].ChoiceSols[1].C
        series.append((m, c, np.gradient(c, m), color, ls))
    return series


def plot_consumption(models, colors, linestyles, legend, title, savepath, t=15):
    # Worker consumption. Writes the paper PDF only.
    fig, ax = plt.subplots()
    for m, c, _, color, ls in _series(models, colors, linestyles, t):
        ax.plot(m, c, color=color, linestyle="-", linewidth=1.6)
    ax.set_xlim([15, 120])
    ax.set_ylim([15, 25])
    ax.set_xlabel("Resources, $m_t$")
    ax.set_ylabel("Consumption, $C(m_t)$")
    ax.set_title(title, pad=22)
    ax.set_yticks(np.arange(15, 26, step=1.0))
    ax.legend(
        legend, loc="lower center", bbox_to_anchor=(0.5, 1.0),
        ncol=len(legend), frameon=False, borderaxespad=0.25,
    )
    fig.tight_layout()
    fig.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.show()


def plot_mpc(models, colors, linestyles, legend, title, savepath, t=15):
    # Marginal propensity. Writes savepath with _MPC before the extension.
    fig, ax = plt.subplots()
    for m, _, mpc, color, ls in _series(models, colors, linestyles, t):
        ax.plot(m, mpc, color=color, linestyle="-", linewidth=1.6)
    ax.set_xlim([15, 120])
    ax.set_ylim([-1.5, 1.5])
    ax.set_xlabel("Resources, $m_t$")
    ax.set_ylabel(r"Marginal Propensity to Consume, $\partial C / \partial m$")
    ax.set_title(title, pad=22)
    ax.legend(
        legend, loc="lower center", bbox_to_anchor=(0.5, 1.0),
        ncol=len(legend), frameon=False, borderaxespad=0.25,
    )
    fig.tight_layout()
    fig.savefig(str(savepath).replace(".pdf", "_MPC.pdf"), dpi=150, bbox_inches="tight")
    plt.show()


In [2]:
CRRA = 1.0
DiscFac = 0.98
Rfree = 1.0
DisUtil = 1.0
T = 20
sigma_s = 0.0
sigma_t = 0.0
sigma_xi = 0.0
sigma_phi = 0.0
prob_opt = 1.0
smoothtype = 'type1'

aXtraMin = 1e-6
aXtraMax = 400.0
aXtraCount = 6000
aXtraExtra = ()  # additional points around known problematic areas
aXtraNestFac = 1

YWork = 20.0
YRet = 0.0
TranShkStd = [0.000] * T
TranShkCount = 1
PermShkStd = [0.0] * T
PermShkCount = 1
TasteShkCount = 1
PhiShkCount = 1
EGMCount = 7000
saveCommon = False
T_retire = 0  # not applicable; retirement is endogenous
T_cycle = T
LivPrb = [1.0] * T
PermGroFac = [1.0] * T
UnempPrb = 0.0
UnempPrbRet = 0.0
IncUnemp = 0.0
IncUnempRet = 0.0

retiring_params = {
    'CRRA': CRRA,
    'Rfree': Rfree,
    'DiscFac': DiscFac,
    'DisUtil': DisUtil,
    'T': T,
    'sigma_s': sigma_s,
    'sigma_t': sigma_t,
    'sigma_xi': sigma_xi,
    'sigma_phi': sigma_phi,
    'prob_opt': prob_opt,
    'smoothtype': smoothtype,
    'aXtraMin': aXtraMin,
    'aXtraMax': aXtraMax,
    'aXtraCount': aXtraCount,
    'aXtraExtra': aXtraExtra,
    'aXtraNestFac': aXtraNestFac,
    'UnempPrb': UnempPrb,
    'UnempPrbRet': UnempPrbRet,
    'IncUnemp': IncUnemp,
    'IncUnempRet': IncUnempRet,
    'YWork': YWork,
    'YRet': YRet,
    'TranShkStd': TranShkStd,
    'TranShkCount': TranShkCount,
    'PermShkStd': PermShkStd,
    'PermShkCount': PermShkCount,
    'TasteShkCount': TasteShkCount,
    'PhiShkCount': PhiShkCount,
    'EGMCount': EGMCount,
    'T_retire': T_retire,
    'T_cycle': T_cycle,
    'LivPrb': LivPrb,
    'PermGroFac': PermGroFac,
    'saveCommon': saveCommon,
}

## Paper figures

The four figures in the retirement section, in the order the text uses them.
Each call writes one consumption PDF. Marginal propensities are in the additional section.

| Text | PDF | What varies |
| --- | --- | --- |
| `fig:ev` | `EV_VaryT.pdf` | Extreme Value scale $\sigma_\zeta \in \{0, 0.05, 0.10, 0.15\}$ |
| `fig:search` | `Search_VaryS.pdf` | Search cost $\sigma_s \in \{0, 0.15, 0.50, 1\}$ |
| `fig:price` (a) | `Price_VaryPhi.pdf` | Account dispersion $\sigma_\phi$, $N_\phi = 151$ |
| `fig:price` (b) | `Price_VaryPhiCount.pdf` | Number of account nodes, $\sigma_\phi = 0.1$ |
| `fig:searchphi` (a) | `SearchPhi_VaryS.pdf` | Search cost, $\sigma_\phi = 0.1$ held fixed |
| `fig:searchphi` (b) | `SearchPhi_VaryPhi.pdf` | Account dispersion, $\sigma_s = 0.5$ held fixed |

### 1. Extreme Value Type I

`EV_VaryT.pdf`, label `fig:ev`. Search and account dispersion are shut down.
The base block's exogenous arrival probability is still inherited here (`prob_opt = 1`).

In [3]:
fig_ev_1_params = copy.deepcopy(retiring_params)
fig_ev_1_params['Rfree'] = 1.0
fig_ev_1_params['DiscFac'] = 0.98
fig_ev_1_params['sigma_t'] = 0.0
fig_ev_1_params['sigma_s'] = 0.0
fig_ev_1_params['sigma_phi'] = 0.0
fig_ev_1_params['PhiShkCount'] = 1
fig_ev_1_params['TranShkStd'] = [np.sqrt(0.000)] * fig_ev_1_params['T']
fig_ev_1_params['TranShkCount'] = 1
modelfig_ev_1 = RetiringDeaton(**fig_ev_1_params)
modelfig_ev_1.solve()

fig_ev_2_params = copy.deepcopy(fig_ev_1_params)
fig_ev_2_params['sigma_t'] = 0.05
modelfig_ev_2 = RetiringDeaton(**fig_ev_2_params)
modelfig_ev_2.solve()

fig_ev_3_params = copy.deepcopy(fig_ev_1_params)
fig_ev_3_params['sigma_t'] = 0.1
modelfig_ev_3 = RetiringDeaton(**fig_ev_3_params)
modelfig_ev_3.solve()

fig_ev_4_params = copy.deepcopy(fig_ev_1_params)
fig_ev_4_params['sigma_t'] = 0.15
modelfig_ev_4 = RetiringDeaton(**fig_ev_4_params)
modelfig_ev_4.solve()

plot_consumption(
    [modelfig_ev_1, modelfig_ev_2, modelfig_ev_3, modelfig_ev_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$\sigma_{\zeta} = 0$", r"$\sigma_{\zeta} = 0.05$", r"$\sigma_{\zeta} = 0.10$", r"$\sigma_{\zeta} = 0.15$"],
    r"$\sigma_{\zeta} \in \{0, 0.05, 0.10, 0.15\}$, $\sigma_s = 0$, $\sigma_{\phi} = 0$",
    "../Figures/Figures_v4/EV_VaryT.pdf",
    t=15,
)


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:44: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 2. Search alone

`Search_VaryS.pdf`, label `fig:search`. Taste shocks and account dispersion are shut down.
The option arrives only through search (`prob_opt = 0`).

In [4]:
fig_s_1_params = copy.deepcopy(retiring_params)
fig_s_1_params['Rfree'] = 1.0
fig_s_1_params['DiscFac'] = 0.98
fig_s_1_params['prob_opt'] = 0.0
fig_s_1_params['sigma_t'] = 0.0
fig_s_1_params['sigma_s'] = 0.0
fig_s_1_params['sigma_phi'] = 0.0
fig_s_1_params['PhiShkCount'] = 1
fig_s_1_params['TranShkStd'] = [np.sqrt(0.000)] * fig_s_1_params['T']
fig_s_1_params['TranShkCount'] = 1
modelfig_s_1 = RetiringDeaton(**fig_s_1_params)
modelfig_s_1.solve()

fig_s_2_params = copy.deepcopy(fig_s_1_params)
fig_s_2_params['sigma_s'] = 0.15
modelfig_s_2 = RetiringDeaton(**fig_s_2_params)
modelfig_s_2.solve()

fig_s_3_params = copy.deepcopy(fig_s_1_params)
fig_s_3_params['sigma_s'] = 0.5
modelfig_s_3 = RetiringDeaton(**fig_s_3_params)
modelfig_s_3.solve()

fig_s_4_params = copy.deepcopy(fig_s_1_params)
fig_s_4_params['sigma_s'] = 1.0
modelfig_s_4 = RetiringDeaton(**fig_s_4_params)
modelfig_s_4.solve()

plot_consumption(
    [modelfig_s_1, modelfig_s_2, modelfig_s_3, modelfig_s_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$\sigma_s = 0$", r"$\sigma_s = 0.15$", r"$\sigma_s = 0.50$", r"$\sigma_s = 1.00$"],
    r"$\sigma_s$ = Varying, $\sigma_{\zeta} = 0.0$, $\sigma_{\phi} = 0.0$",
    "../Figures/Figures_v4/Search_VaryS.pdf",
    t=15,
)


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:44: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 3. Account dispersion alone

Label `fig:price`. Search and taste shocks are shut down, so the account draw arrives with certainty.
Panel (a) varies $\sigma_\phi$. Panel (b) holds $\sigma_\phi = 0.1$ and varies the number of nodes.
The base block's exogenous arrival probability is still inherited here (`prob_opt = 1`).

In [5]:
# Panel (a): Price_VaryPhi.pdf. Vary sigma_phi at 151 nodes.
fig_phi_1_params = copy.deepcopy(retiring_params)
fig_phi_1_params['Rfree'] = 1.0
fig_phi_1_params['DiscFac'] = 0.98
fig_phi_1_params['sigma_t'] = 0.0
fig_phi_1_params['sigma_s'] = 0.0
fig_phi_1_params['TranShkStd'] = [0.0] * fig_phi_1_params['T']
fig_phi_1_params['TranShkCount'] = 1
fig_phi_1_params['sigma_phi'] = 0.0
fig_phi_1_params['PhiShkCount'] = 1
modelfig_phi_1 = RetiringDeaton(**fig_phi_1_params)
modelfig_phi_1.solve()

fig_phi_2_params = copy.deepcopy(fig_phi_1_params)
fig_phi_2_params['sigma_phi'] = 0.05
fig_phi_2_params['PhiShkCount'] = 151
modelfig_phi_2 = RetiringDeaton(**fig_phi_2_params)
modelfig_phi_2.solve()

fig_phi_3_params = copy.deepcopy(fig_phi_1_params)
fig_phi_3_params['sigma_phi'] = 0.1
fig_phi_3_params['PhiShkCount'] = 151
modelfig_phi_3 = RetiringDeaton(**fig_phi_3_params)
modelfig_phi_3.solve()

fig_phi_4_params = copy.deepcopy(fig_phi_1_params)
fig_phi_4_params['sigma_phi'] = 0.15
fig_phi_4_params['PhiShkCount'] = 151
modelfig_phi_4 = RetiringDeaton(**fig_phi_4_params)
modelfig_phi_4.solve()

plot_consumption(
    [modelfig_phi_1, modelfig_phi_2, modelfig_phi_3, modelfig_phi_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$\sigma_{\phi} = 0$", r"$\sigma_{\phi} = 0.05$", r"$\sigma_{\phi} = 0.10$", r"$\sigma_{\phi} = 0.15$"],
    r"$\sigma_{\phi}$ = Varying, $N_{\phi} = 151$, $\sigma_s = 0.0$, $\sigma_{\zeta} = 0.0$",
    "../Figures/Figures_v4/Price_VaryPhi.pdf",
    t=15,
)


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:44: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [6]:
# Panel (b): Price_VaryPhiCount.pdf. Fix sigma_phi = 0.1, vary the number of nodes.
fig_nphi_1_params = copy.deepcopy(retiring_params)
fig_nphi_1_params['Rfree'] = 1.0
fig_nphi_1_params['DiscFac'] = 0.98
fig_nphi_1_params['sigma_t'] = 0.0
fig_nphi_1_params['sigma_s'] = 0.0
fig_nphi_1_params['TranShkStd'] = [0.0] * fig_nphi_1_params['T']
fig_nphi_1_params['TranShkCount'] = 1
fig_nphi_1_params['sigma_phi'] = 0.1
fig_nphi_1_params['PhiShkCount'] = 1
modelfig_nphi_1 = RetiringDeaton(**fig_nphi_1_params)
modelfig_nphi_1.solve()

fig_nphi_2_params = copy.deepcopy(fig_nphi_1_params)
fig_nphi_2_params['PhiShkCount'] = 9
modelfig_nphi_2 = RetiringDeaton(**fig_nphi_2_params)
modelfig_nphi_2.solve()

fig_nphi_3_params = copy.deepcopy(fig_nphi_1_params)
fig_nphi_3_params['PhiShkCount'] = 151
modelfig_nphi_3 = RetiringDeaton(**fig_nphi_3_params)
modelfig_nphi_3.solve()

fig_nphi_4_params = copy.deepcopy(fig_nphi_1_params)
fig_nphi_4_params['PhiShkCount'] = 351
modelfig_nphi_4 = RetiringDeaton(**fig_nphi_4_params)
modelfig_nphi_4.solve()

plot_consumption(
    [modelfig_nphi_1, modelfig_nphi_2, modelfig_nphi_3, modelfig_nphi_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$N_{\phi} = 1$", r"$N_{\phi} = 9$", r"$N_{\phi} = 151$", r"$N_{\phi} = 351$"],
    r"$\sigma_{\phi} = 0.1$, $N_{\phi}$ = Varying, $\sigma_s = 0.0$, $\sigma_{\zeta} = 0.0$",
    "../Figures/Figures_v4/Price_VaryPhiCount.pdf",
    t=15,
)


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:44: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 4. Search and the account factor

Label `fig:searchphi`. Taste shocks are shut down. The option arrives only through search (`prob_opt = 0`).
Panel (a) varies $\sigma_s$ at $\sigma_\phi = 0.1$. Panel (b) varies $\sigma_\phi$ at $\sigma_s = 0.5$.

In [7]:
# Panel (a): SearchPhi_VaryS.pdf. sigma_phi = 0.1, N_phi = 151, vary sigma_s.
fig_sp_s_1_params = copy.deepcopy(retiring_params)
fig_sp_s_1_params['Rfree'] = 1.0
fig_sp_s_1_params['DiscFac'] = 0.98
fig_sp_s_1_params['prob_opt'] = 0.0
fig_sp_s_1_params['sigma_t'] = 0.0
fig_sp_s_1_params['sigma_s'] = 0.0
fig_sp_s_1_params['TranShkStd'] = [0.0] * fig_sp_s_1_params['T']
fig_sp_s_1_params['TranShkCount'] = 1
fig_sp_s_1_params['sigma_phi'] = 0.1
fig_sp_s_1_params['PhiShkCount'] = 151
modelfig_sp_s_1 = RetiringDeaton(**fig_sp_s_1_params)
modelfig_sp_s_1.solve()

fig_sp_s_2_params = copy.deepcopy(fig_sp_s_1_params)
fig_sp_s_2_params['sigma_s'] = 0.15
modelfig_sp_s_2 = RetiringDeaton(**fig_sp_s_2_params)
modelfig_sp_s_2.solve()

fig_sp_s_3_params = copy.deepcopy(fig_sp_s_1_params)
fig_sp_s_3_params['sigma_s'] = 0.5
modelfig_sp_s_3 = RetiringDeaton(**fig_sp_s_3_params)
modelfig_sp_s_3.solve()

fig_sp_s_4_params = copy.deepcopy(fig_sp_s_1_params)
fig_sp_s_4_params['sigma_s'] = 1.0
modelfig_sp_s_4 = RetiringDeaton(**fig_sp_s_4_params)
modelfig_sp_s_4.solve()

plot_consumption(
    [modelfig_sp_s_1, modelfig_sp_s_2, modelfig_sp_s_3, modelfig_sp_s_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$\sigma_s = 0$", r"$\sigma_s = 0.15$", r"$\sigma_s = 0.50$", r"$\sigma_s = 1.00$"],
    r"$\sigma_s$ = Varying, $\sigma_{\phi} = 0.1$, $N_{\phi} = 151$, $\sigma_{\zeta} = 0.0$",
    "../Figures/Figures_v4/SearchPhi_VaryS.pdf",
    t=15,
)


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:44: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [8]:
# Panel (b): SearchPhi_VaryPhi.pdf. sigma_s = 0.5, N_phi = 151, vary sigma_phi.
fig_sp_phi_1_params = copy.deepcopy(retiring_params)
fig_sp_phi_1_params['Rfree'] = 1.0
fig_sp_phi_1_params['DiscFac'] = 0.98
fig_sp_phi_1_params['prob_opt'] = 0.0
fig_sp_phi_1_params['sigma_t'] = 0.0
fig_sp_phi_1_params['sigma_s'] = 0.5
fig_sp_phi_1_params['TranShkStd'] = [0.0] * fig_sp_phi_1_params['T']
fig_sp_phi_1_params['TranShkCount'] = 1
fig_sp_phi_1_params['sigma_phi'] = 0.0
fig_sp_phi_1_params['PhiShkCount'] = 1
modelfig_sp_phi_1 = RetiringDeaton(**fig_sp_phi_1_params)
modelfig_sp_phi_1.solve()

fig_sp_phi_2_params = copy.deepcopy(fig_sp_phi_1_params)
fig_sp_phi_2_params['sigma_phi'] = 0.05
fig_sp_phi_2_params['PhiShkCount'] = 151
modelfig_sp_phi_2 = RetiringDeaton(**fig_sp_phi_2_params)
modelfig_sp_phi_2.solve()

fig_sp_phi_3_params = copy.deepcopy(fig_sp_phi_1_params)
fig_sp_phi_3_params['sigma_phi'] = 0.1
fig_sp_phi_3_params['PhiShkCount'] = 151
modelfig_sp_phi_3 = RetiringDeaton(**fig_sp_phi_3_params)
modelfig_sp_phi_3.solve()

fig_sp_phi_4_params = copy.deepcopy(fig_sp_phi_1_params)
fig_sp_phi_4_params['sigma_phi'] = 0.15
fig_sp_phi_4_params['PhiShkCount'] = 151
modelfig_sp_phi_4 = RetiringDeaton(**fig_sp_phi_4_params)
modelfig_sp_phi_4.solve()

plot_consumption(
    [modelfig_sp_phi_1, modelfig_sp_phi_2, modelfig_sp_phi_3, modelfig_sp_phi_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$\sigma_{\phi} = 0$", r"$\sigma_{\phi} = 0.05$", r"$\sigma_{\phi} = 0.10$", r"$\sigma_{\phi} = 0.15$"],
    r"$\sigma_s = 0.5$, $\sigma_{\phi}$ = Varying, $N_{\phi} = 151$, $\sigma_{\zeta} = 0.0$",
    "../Figures/Figures_v4/SearchPhi_VaryPhi.pdf",
    t=15,
)


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:44: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Direct comparison

`EV_vs_SearchPhi_EV015.pdf`, for a trial of the direct comparison at $\sigma_\zeta = 0.15$. The saved pair for $\sigma_\zeta = 0.10$ stays in `Output/Retirement_EV_vs_SearchPhi.pkl` and is not overwritten.

The target is the Extreme Value consumption function at $\sigma_\zeta = 0.15$ (`modelfig_ev_4`), at $t = 15$. Search and account dispersion are shut down on that solve. The free parameters are $\sigma_s$ and $\sigma_\phi$, with taste shocks off, 151 account nodes, and the option arriving only through search.

The loss is the root mean squared gap on a uniform grid from 20 to 40, the resources where the unsmoothed rule is discontinuous. The figure still shows 15 to 120. The gap from 40 to 120 is printed and is not the objective. Set `optimize = True` to run the estimation. `scipy.optimize.brentq` finds the minimum of that loss by solving its first-order condition: the slope in one parameter, holding the other fixed. The two parameters alternate until both stop moving. $\sigma_s$ is kept in $[0, 2]$ and $\sigma_\phi$ in $[0, 0.30]$. The minimizing pair is written to `Output/Retirement_EV_vs_SearchPhi_EV015.pkl`.

A pickle already in that file was fit on resources from 15 to 120. Set `optimize = True` and run the cell again so the saved pair is the minimum on 20 to 40.

With `optimize = False` the estimation is skipped. The figure is drawn from the saved parameters. The minimized distance is positive. The figure shows how close the two rules get.

The Extreme Value target still inherits `prob_opt = 1` from the base block. The search-and-account solves set `prob_opt = 0`.

In [9]:
# True: estimate sigma_s and sigma_phi, then save them.
# False: skip the estimation and draw the figure from the saved parameters.
optimize = False
fit_path = Path("../Output/Retirement_EV_vs_SearchPhi_EV015.pkl")

In [10]:
def _worker_c(model, t):
    m = np.asarray(model.mGrid, dtype=float)
    c = np.asarray(model.solution[t].ChoiceSols[1].C, dtype=float)
    order = np.argsort(m)
    return m[order], c[order]


m_fit = np.linspace(20.0, 40.0, 200)
m_tail = np.linspace(40.0, 120.0, 320)
_m_worker, _c_worker = _worker_c(modelfig_ev_4, t=15)
target_fit = np.interp(m_fit, _m_worker, _c_worker)
target_tail = np.interp(m_tail, _m_worker, _c_worker)

# Pairs already solved in the paper figures. Keys are (sigma_s, sigma_phi).
pair_cache = {
    (0.0, 0.0): modelfig_s_1,
    (0.0, 0.1): modelfig_sp_s_1,
    (0.15, 0.1): modelfig_sp_s_2,
    (0.5, 0.1): modelfig_sp_s_3,
    (1.0, 0.1): modelfig_sp_s_4,
    (0.5, 0.0): modelfig_sp_phi_1,
    (0.5, 0.05): modelfig_sp_phi_2,
    (0.5, 0.15): modelfig_sp_phi_4,
}

S_LO, S_HI = 0.0, 2.0
PHI_LO, PHI_HI = 0.0, 0.30


def _project(x):
    sigma_s = float(np.clip(x[0], S_LO, S_HI))
    sigma_phi = float(np.clip(x[1], PHI_LO, PHI_HI))
    if sigma_phi < 1e-10:
        sigma_phi = 0.0
    return np.array([sigma_s, sigma_phi])


def _cache_key(sigma_s, sigma_phi):
    return (round(float(sigma_s), 6), round(float(sigma_phi), 6))


def solve_pair(sigma_s, sigma_phi):
    sigma_s, sigma_phi = _project([sigma_s, sigma_phi])
    key = _cache_key(sigma_s, sigma_phi)
    if key in pair_cache:
        return pair_cache[key]
    params = copy.deepcopy(fig_sp_s_1_params)
    params["sigma_s"] = float(sigma_s)
    params["sigma_phi"] = float(sigma_phi)
    params["PhiShkCount"] = 1 if sigma_phi == 0.0 else 151
    print(f"solving  sigma_s={sigma_s:.4f}  sigma_phi={sigma_phi:.4f}")
    model = RetiringDeaton(**params)
    model.solve()
    pair_cache[key] = model
    return model


def _rms_on(model, m_grid, target):
    fitted = np.interp(m_grid, *_worker_c(model, t=15))
    return float(np.sqrt(np.mean((fitted - target) ** 2)))


def pair_rms(model):
    return _rms_on(model, m_fit, target_fit)


def tail_rms(model):
    return _rms_on(model, m_tail, target_tail)


import pickle

def loss(x):
    return pair_rms(solve_pair(x[0], x[1]))


# brentq solves one scalar equation. The minimum of the distance is where the
# slope in a parameter crosses from negative to positive. Each call holds the
# other parameter fixed. h is the step used to measure that slope.
bounds = (np.array([S_LO, PHI_LO]), np.array([S_HI, PHI_HI]))
slope_step = np.array([0.05, 0.01])
root_tol = np.array([0.02, 0.005])
names = ["sigma_s", "sigma_phi"]


def slope(value, index, theta):
    h = slope_step[index]
    up = np.array(theta, dtype=float)
    dn = np.array(theta, dtype=float)
    up[index] = value + h
    dn[index] = value - h
    return loss(up) - loss(dn)


def loss_at(value, index, theta):
    x = np.array(theta, dtype=float)
    x[index] = value
    return loss(x)


def minimize_coordinate(index, theta):
    h = slope_step[index]
    lo = bounds[0][index] + h
    hi = bounds[1][index] - h
    probes = np.linspace(lo, hi, 5)
    slopes = [slope(value, index, theta) for value in probes]
    print(f"  {names[index]} slopes, other parameter held fixed")
    for value, sl in zip(probes, slopes):
        print(f"    {names[index]}={value:.4f}  slope={sl:+.6f}")

    candidates = [bounds[0][index], bounds[1][index]]
    for a, b, fa, fb in zip(probes[:-1], probes[1:], slopes[:-1], slopes[1:]):
        if fa == 0.0:
            candidates.append(float(a))
        if fa < 0.0 < fb:
            root = brentq(
                lambda value: slope(value, index, theta),
                a,
                b,
                xtol=root_tol[index],
                maxiter=12,
                disp=False,
            )
            candidates.append(float(root))
            print(f"    brentq root {names[index]}={root:.4f}")
    best_value = min(candidates, key=lambda value: loss_at(value, index, theta))
    if best_value in (bounds[0][index], bounds[1][index]):
        print(f"    minimum of {names[index]} is on the bound {best_value:.4f}")
    return float(best_value)


if optimize:
    from scipy.optimize import brentq

    theta = np.array([0.5, 0.1])
    print(
        f"start: sigma_s={theta[0]:.4f}, sigma_phi={theta[1]:.4f}, rms={loss(theta):.5f}"
    )
    for sweep in range(6):
        previous = theta.copy()
        for index in range(2):
            theta[index] = minimize_coordinate(index, theta)
            print(
                f"sweep {sweep}: sigma_s={theta[0]:.4f}, sigma_phi={theta[1]:.4f}, "
                f"rms={loss(theta):.5f}"
            )
        if np.all(np.abs(theta - previous) <= root_tol):
            break

    best_s, best_phi = _project(theta)
    best_model = solve_pair(best_s, best_phi)
    best_rms = pair_rms(best_model)
    best_tail = tail_rms(best_model)
    payload = {
        "sigma_s": float(best_s),
        "sigma_phi": float(best_phi),
        "rms": float(best_rms),
        "rms_tail": float(best_tail),
        "m_fit_lo": 20.0,
        "m_fit_hi": 40.0,
        "sigma_zeta": 0.15,
        "t": 15,
    }
    fit_path.parent.mkdir(parents=True, exist_ok=True)
    with open(fit_path, "wb") as f:
        pickle.dump(payload, f)
    print(
        f"minimum: sigma_s={best_s:.4f}, sigma_phi={best_phi:.4f}, "
        f"rms_20_40={best_rms:.5f}, rms_above_40={best_tail:.5f}"
    )
    print(f"saved {fit_path}")
else:
    if not fit_path.exists():
        raise FileNotFoundError(
            f"No saved parameters at {fit_path}. Set optimize = True and run this cell once."
        )
    with open(fit_path, "rb") as f:
        saved = pickle.load(f)
    best_s = float(saved["sigma_s"])
    best_phi = float(saved["sigma_phi"])
    best_model = solve_pair(best_s, best_phi)
    best_rms = pair_rms(best_model)
    best_tail = tail_rms(best_model)
    print(
        f"loaded {fit_path}: sigma_s={best_s:.4f}, sigma_phi={best_phi:.4f}, "
        f"rms_20_40={best_rms:.5f}, rms_above_40={best_tail:.5f}"
    )


solving  sigma_s=2.0000  sigma_phi=0.2047


loaded ..\Output\Retirement_EV_vs_SearchPhi_EV015.pkl: sigma_s=2.0000, sigma_phi=0.2047, rms_20_40=0.25297, rms_above_40=2.05123


In [11]:
m_ev, c_ev = _worker_c(modelfig_ev_4, t=15)
m_hat, c_hat = _worker_c(best_model, t=15)

fig, ax = plt.subplots()
ax.plot(m_ev, c_ev, color="r", linestyle="-", linewidth=1.6, label=r"Extreme Value, $\sigma_{\zeta} = 0.15$")
ax.plot(
    m_hat,
    c_hat,
    color="k",
    linestyle="--",
    linewidth=1.6,
    label=rf"Search and account, $\sigma_s = {best_s:.2f}$, $\sigma_{{\phi}} = {best_phi:.2f}$",
)
ax.set_xlim([15, 120])
ax.set_ylim([15, 25])
ax.set_xlabel(r"Resources, $m_t$")
ax.set_ylabel(r"Consumption, $C(m_t)$")
ax.set_title(r"Extreme Value and search with the account factor", pad=20)
ax.set_yticks(np.arange(15, 26, step=1.0))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, borderaxespad=0.25)

fig.tight_layout()
fig.savefig("../Figures/Figures_v4/EV_vs_SearchPhi_EV015.pdf", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\358241170.py:24: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Additional figures

Not included in `Text/RetirementModel.tex`. These cells reuse the solves above,
except the base-block solve, which does not write a PDF.

### Marginal propensities

One PDF beside each paper consumption figure, with `_MPC` before the extension.
The text does not include these.

In [12]:
plot_mpc(
    [modelfig_ev_1, modelfig_ev_2, modelfig_ev_3, modelfig_ev_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$\sigma_{\zeta} = 0$", r"$\sigma_{\zeta} = 0.05$", r"$\sigma_{\zeta} = 0.10$", r"$\sigma_{\zeta} = 0.15$"],
    r"$\sigma_{\zeta} \in \{0, 0.05, 0.10, 0.15\}$, $\sigma_s = 0$, $\sigma_{\phi} = 0$",
    "../Figures/Figures_v4/EV_VaryT.pdf",
    t=15,
)
plot_mpc(
    [modelfig_s_1, modelfig_s_2, modelfig_s_3, modelfig_s_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$\sigma_s = 0$", r"$\sigma_s = 0.15$", r"$\sigma_s = 0.50$", r"$\sigma_s = 1.00$"],
    r"$\sigma_s$ = Varying, $\sigma_{\zeta} = 0.0$, $\sigma_{\phi} = 0.0$",
    "../Figures/Figures_v4/Search_VaryS.pdf",
    t=15,
)
plot_mpc(
    [modelfig_phi_1, modelfig_phi_2, modelfig_phi_3, modelfig_phi_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$\sigma_{\phi} = 0$", r"$\sigma_{\phi} = 0.05$", r"$\sigma_{\phi} = 0.10$", r"$\sigma_{\phi} = 0.15$"],
    r"$\sigma_{\phi}$ = Varying, $N_{\phi} = 151$, $\sigma_s = 0.0$, $\sigma_{\zeta} = 0.0$",
    "../Figures/Figures_v4/Price_VaryPhi.pdf",
    t=15,
)
plot_mpc(
    [modelfig_nphi_1, modelfig_nphi_2, modelfig_nphi_3, modelfig_nphi_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$N_{\phi} = 1$", r"$N_{\phi} = 9$", r"$N_{\phi} = 151$", r"$N_{\phi} = 351$"],
    r"$\sigma_{\phi} = 0.1$, $N_{\phi}$ = Varying, $\sigma_s = 0.0$, $\sigma_{\zeta} = 0.0$",
    "../Figures/Figures_v4/Price_VaryPhiCount.pdf",
    t=15,
)
plot_mpc(
    [modelfig_sp_s_1, modelfig_sp_s_2, modelfig_sp_s_3, modelfig_sp_s_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$\sigma_s = 0$", r"$\sigma_s = 0.15$", r"$\sigma_s = 0.50$", r"$\sigma_s = 1.00$"],
    r"$\sigma_s$ = Varying, $\sigma_{\phi} = 0.1$, $N_{\phi} = 151$, $\sigma_{\zeta} = 0.0$",
    "../Figures/Figures_v4/SearchPhi_VaryS.pdf",
    t=15,
)
plot_mpc(
    [modelfig_sp_phi_1, modelfig_sp_phi_2, modelfig_sp_phi_3, modelfig_sp_phi_4],
    ["#000000", "#0072B2", "#D55E00", "#009E73"],
    ["-", "-", "-", "-"],
    [r"$\sigma_{\phi} = 0$", r"$\sigma_{\phi} = 0.05$", r"$\sigma_{\phi} = 0.10$", r"$\sigma_{\phi} = 0.15$"],
    r"$\sigma_s = 0.5$, $\sigma_{\phi}$ = Varying, $N_{\phi} = 151$, $\sigma_{\zeta} = 0.0$",
    "../Figures/Figures_v4/SearchPhi_VaryPhi.pdf",
    t=15,
)


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:63: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:63: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:63: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:63: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:63: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\amonninger\AppData\Local\Temp\ipykernel_20784\3800824481.py:63: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Base-block solve

Solves `retiring_params` as written above and does not write a PDF.
All scales are zero and `prob_opt = 1`. This is not a paper figure.

In [13]:
# model = RetiringDeaton(**retiring_params)
# model.solve()